# NHS Adult Autism Assessment Pathway DES Model

Interactive demonstration of the **`adhd_simpy`** discrete-event simulation package.

**Prerequisites:** `pip install -e .` from the project root.

Run all cells top to bottom (`Kernel → Restart & Run All`).

## Table of Contents

1. [Background](#1-background)
2. [Project Aims](#2-project-aims)
3. [Proposed Solution](#3-proposed-solution)
4. [Discrete Event Simulation (DES) Model](#4-discrete-event-simulation-des-model)
   - 4.1 [Overview](#41-overview-of-the-des-model)
   - 4.2 [Default Parameters](#42-default-parameters)
   - 4.3 [Experiment Class](#43-experiment-class)
   - 4.4 [Patient Class](#44-patient-class)
   - 4.5 [Workforce Resource](#45-workforce-resource-and-resource-constraints)
   - 4.6 [Autism Pathway System Class](#46-autism-pathway-system-class)
5. [Classes and Functions](#5-classes-and-functions-of-the-autism-pathway-model)
6. [Simulation Execution](#6-simulation-execution)
   - 6.1 [Single Trace Run](#61-single-trace-run)
   - 6.2 [Single Simulation Run](#62-single-simulation-run-with-and-without-warm-up)
   - 6.3 [Multiple Replications](#63-multiple-replications-fixed-and-random-seeds)
7. [Model Applications](#7-model-applications)
   - 7.1–7.6 Baseline, experiments, warm-up, calibration, freeze-state, scenarios
8. [Model Verification and Validation](#8-model-verification-and-validation)
9. [Future Work](#9-future-work)


---

# 1. Background

Adult Autism Assessment services across the NHS experience substantial waiting lists and increasing referral demand. Many services operate under limited clinical capacity, resulting in long Referral-to-Treatment (RTT) times, growing assessment backlogs, and increasing pressure on specialist clinicians.

Evaluating operational interventions directly within NHS services is difficult because of financial costs, service disruption, and clinical risk.

Discrete Event Simulation (DES) provides a virtual environment where service behaviour can be reproduced, analysed, and tested before operational changes are implemented.


---

# 2. Project Aims

The objectives of this project are to:

- Develop a realistic DES model of the NHS Adult Autism Assessment pathway.
- Model workforce resource constraints using clinician working hours.
- Represent patient flow from referral to discharge.
- Capture waiting lists and Referral-to-Treatment (RTT) performance.
- Support configurable NHS providers using external configuration files.
- Produce realistic operational system states.
- Support intervention and scenario analysis.
- Provide a decision-support tool for service planning.


---

# 3. Proposed Solution

The project proposes a configurable Discrete Event Simulation framework built using **SimPy**.

The framework consists of:

- Patient-level simulation
- Workforce-hour constrained resources
- NHS autism pathway representation
- Warm-up and steady-state simulation
- Dynamic provider calibration *(planned — see §7.4)*
- Frozen operational state generation *(planned — see §7.5)*
- Scenario analysis framework

The same simulation engine can therefore be configured for multiple NHS providers without modifying the DES logic. Provider-specific settings live in modules such as `adhd_simpy/Model/parameters.py` and `adhd_simpy/Model/devon_parameters.py`.


---

# 4. Discrete Event Simulation (DES) Model

## 4.1 Overview of the DES Model

- Built using **SimPy**.
- Event-driven patient simulation.
- Individual patient entities.
- Multiple clinical pathway stages.
- Workforce-hour constrained clinical resources.
- Priority and standard waiting queues.
- Daily replenishment of clinician hours.
- **Warm-up → collection → drain** phases.
- Provider-configurable parameters.

### Patient flow

Each patient is a SimPy process. Referrals arrive on **weekdays only**. Capacity-constrained stages queue for clinician **hours**, not fixed slots.



#### Figure 1 — Main pathway
![Figure 1: Patient pathway](assets/demo/01_patient_pathway.png)

#### Figure 2 — Clinical stage 
![Figure 2: Clinical stage](assets/demo/02_clinical_stage.png)

#### Figure 3 — Workforce
![Figure 3: Review loop](assets/demo/03_workforce_hours.png)


In [66]:
import pandas as pd
from adhd_simpy.Model import (
    Audit,
    AutismPathwaySystem,
    Experiment,
    Patient,
    WorkforceHoursResource,
    multiple_runs,
    single_run,
)
from adhd_simpy.Model.parameters import (
    DEFAULT_RND_SET,
    MAX_DRAIN_DAYS,
    N_REP,
    REFERRALS_PER_DAY,
    RUN_LENGTH,
    WARMUP_DAYS,
)


## 4.2 Default Parameters

Default values are defined in `adhd_simpy/Model/parameters.py`.

| Category | Parameters | Description |
|----------|------------|-------------|
| **Simulation Control** | `RUN_LENGTH`, `WARMUP_DAYS`, `COOLDOWN_DAYS`, `NUMBER_OF_RUNS`, `TRACE`, `DEFAULT_SEED` | Controls simulation duration, warm-up, replications, tracing and random seed. |
| **Referral Generation** | `MEAN_REFERRAL_INTERVAL`, `PRIORITY_REFERRAL_PROPORTION` | Controls referral arrival process and priority referral proportion. |
| **Clinical Pathway Probabilities** | `P_TRIAGE_REJECTION`, `P_SCREENING_DISCHARGE`, `P_PREASSESSMENT_REJECTION`, `P_FURTHER_ASSESSMENT`, `P_AUTISM_DIAGNOSIS`, `P_POST_DIAG_SUPPORT`, `P_REVIEW`, `P_SELF_DISCHARGE` | Defines branching probabilities throughout the autism assessment pathway. |
| **Service Duration Parameters** | `SCREENING_DURATION`, `PREASSESSMENT_DURATION`, `ASSESSMENT_DURATION`, `FURTHER_ASSESSMENT_DURATION`, `POST_DIAG_DURATION`, `OTHER_DURATION`, `REVIEW_DURATION` | Probability distributions defining appointment durations at each clinical stage. |
| **Workforce Capacity** | `SCREENING_WORKFORCE_HOURS`, `PREASSESSMENT_WORKFORCE_HOURS`, `ASSESSMENT_WORKFORCE_HOURS`, `FURTHER_ASSESSMENT_WORKFORCE_HOURS`, `POST_DIAG_WORKFORCE_HOURS`, `OTHER_WORKFORCE_HOURS`, `REVIEW_WORKFORCE_HOURS` | Daily clinician hours available at each pathway stage. |
| **Statistical Configuration** | Collection period, confidence interval settings, random number streams | Controls statistical analysis and reproducibility of simulation experiments. |

These parameters allow the same DES engine to be configured for different NHS providers without modifying the underlying simulation model.

In [53]:
from adhd_simpy.Model import parameters as p

default_params = pd.Series(
    {
        "REFERRALS_PER_DAY": p.REFERRALS_PER_DAY,
        "WARMUP_DAYS": p.WARMUP_DAYS,
        "RUN_LENGTH": p.RUN_LENGTH,
        "MAX_DRAIN_DAYS": p.MAX_DRAIN_DAYS,
        "PCT_REFERRAL_REJECTED": p.PCT_REFERRAL_REJECTED,
        "PCT_NON_DIAGNOSIS_AT_ASSESSMENT": p.PCT_NON_DIAGNOSIS_AT_ASSESSMENT,
        "WORKFORCE_HOURS_ASSESSMENT": p.WORKFORCE_HOURS_ASSESSMENT,
        "PCT_PRIORITY_ASSESSMENT": p.PCT_PRIORITY_ASSESSMENT,
        "DEFAULT_RND_SET": p.DEFAULT_RND_SET,
        "N_REP": p.N_REP,
    },
    name="default",
)
default_params


REFERRALS_PER_DAY                     5.00
WARMUP_DAYS                         730.00
RUN_LENGTH                         1825.00
MAX_DRAIN_DAYS                     3650.00
PCT_REFERRAL_REJECTED                 0.20
PCT_NON_DIAGNOSIS_AT_ASSESSMENT       0.20
WORKFORCE_HOURS_ASSESSMENT           24.00
PCT_PRIORITY_ASSESSMENT               0.15
DEFAULT_RND_SET                      42.00
N_REP                                20.00
Name: default, dtype: float64

## 4.3 Experiment Class

**Module:** `adhd_simpy/Model/experiment.py`

Responsible for:

- Simulation configuration
- Random number management (25 independent streams)
- Parameter loading (via constructor `**kwargs` overrides)
- Results collection (`results` flow counters)
- Seeded distribution objects for arrivals, durations, and branching

```python
experiment = Experiment(auditor=Audit(), random_number_set=42)
experiment = Experiment(auditor=Audit(), iat=1/10, workforce_hours_assessment=36)
```


## 4.4 Patient Class

**Module:** `adhd_simpy/Model/patient.py`

Responsible for modelling:

- Individual referrals as SimPy processes
- Clinical pathway progression (triage → 7 stages → review loop)
- Waiting times and resource requests
- Priority status (re-drawn at each stage)
- Clinical outcomes and flow counters
- RTT collection (cohort-filtered via `collect_stats`)


## 4.5 Workforce Resource and Resource Constraints

**Module:** `adhd_simpy/Model/resources.py` — class `WorkforceHoursResource`

Custom SimPy resource representing NHS clinician capacity. Unlike standard SimPy resources, capacity is measured in **clinician hours** rather than server count.

| Feature | Behaviour |
|---------|----------|
| Capacity | Clinician **hours per weekday** (Mon–Fri); weekends = 0 |
| Queues | Priority deque served before standard |
| Scheduling | **Best-fit** — largest job fitting `hours_left` |
| Accounting | Released / used / unused hours |
| Validation | `final_validate()` — hour balance and queue conservation |


## 4.6 Autism Pathway System Class

**Module:** `adhd_simpy/Model/system.py`

Responsible for:

- Weekday referral generation (exponential inter-arrivals)
- Patient routing and spawning
- One `WorkforceHoursResource` per clinical stage
- Daily queue-length snapshots to `Audit`
- System coordination via `run()` SimPy generator

### Simulation phases

```
Phase 1  WARM-UP     Days 0 → warmup_days           KPI cohort OFF
Phase 2  COLLECTION  Days warmup → + run_length      KPI cohort ON
Phase 3  DRAIN       After last referral until empty (max MAX_DRAIN_DAYS)
```


---

# 5. Classes and Functions of the Autism Pathway Model

| Component | Module | Role |
|-----------|--------|------|
| `Experiment` | `experiment.py` | Configuration, RNG, flow counters |
| `Patient` | `patient.py` | SimPy pathway process |
| `AutismPathwaySystem` | `system.py` | Arrivals, resources, coordination |
| `WorkforceHoursResource` | `resources.py` | Hour-based capacity engine |
| `Audit` | `audit.py` | RTT, wait, queue, utilisation KPIs |
| `single_run` / `multiple_runs` | `simulation.py` | Replication runners |
| Distributions | `distributions.py` | Exponential, Triangular, Bernoulli, Choice |
| Default parameters | `parameters.py` | Global scenario defaults |
| Provider scenario | `devon_parameters.py` | Devon (DAANA) published data |
| Verification | `verification.py` | Automated V&V test functions |
| Dynamic calibration | — | *Planned (§7.4)* |
| State freeze | — | *Planned (§7.5)* |


## 6.1 Single Trace Run

Purpose: debugging, event verification, patient pathway tracing, resource behaviour inspection.

Set `parameters.TRACE = True` before calling `single_run()`. The `trace()` helper reads that flag at **call time** (so toggling it in the notebook works).

> Use a short `run_length` (e.g. 3 days) — trace output is verbose (one line per pathway event).


In [54]:
import adhd_simpy.Model.parameters as params

params.TRACE = True
trace_exp = Experiment(auditor=Audit(), random_number_set=42)
trace_results = single_run(trace_exp, rep=0, warmup_days=0, run_length=3)
params.TRACE = False  # always turn off after tracing

print(
    f"Trace complete: {trace_results['ARRIVED_TOTAL']:.0f} cohort arrivals "
    f"in a 3-day collection window (see pathway messages above)"
)


[Time 0.221 | Monday] Patient 1 entered system. Referral submitted.
[Time 0.221 | Monday] Patient 1 Exit - Referral Rejected at Triage.
[Time 0.360 | Monday] Patient 2 entered system. Referral submitted.
[Time 0.360 | Monday] Patient 2 Exit - Referral Rejected at Triage.
[Time 0.499 | Monday] Patient 3 entered system. Referral submitted.
[Time 0.499 | Monday] Patient 3 queued for SCREENING (Queue pos: 0).
[Time 0.499 | Monday] >> Patient 3 officially ENTERED SCREENING stage.
[Time 0.527 | Monday] Patient 3 queued for PRE-ASSESSMENT (Queue pos: 0).
[Time 0.527 | Monday] >> Patient 3 officially ENTERED PRE-ASSESSMENT stage.
[Time 0.578 | Monday] Patient 3 queued for CORE ASSESSMENT (Queue pos: 0).
[Time 0.578 | Monday] >> Patient 3 officially ENTERED CORE ASSESSMENT stage.
[Time 0.740 | Monday] Patient 3 queued for POST-DIAG OTHER SUPPORT.
[Time 0.740 | Monday] >> Patient 3 officially ENTERED POST-DIAG OTHER SUPPORT stage.
[Time 0.809 | Monday] Patient 3 queued for FINAL CASE DISCHARGE R

## 6.2 Single Simulation Run (With and Without Warm-up)

Supports: no warm-up, fixed warm-up, full patient drain.

Used for baseline analysis, KPI generation, and resource utilisation.


In [55]:
DEMO_RUN_LENGTH = 365

auditor = Audit()
experiment = Experiment(auditor=auditor, random_number_set=DEFAULT_RND_SET)

baseline = single_run(experiment, rep=0, warmup_days=0, run_length=DEMO_RUN_LENGTH)
with_warmup = single_run(
    Experiment(auditor=Audit(), random_number_set=DEFAULT_RND_SET),
    rep=0,
    warmup_days=365,
    run_length=DEMO_RUN_LENGTH,
)

KPI_KEYS = [
    "ARRIVED_TOTAL",
    "FLOW_DIAGNOSIS_CONFIRMED",
    "ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS",
    "QUEUE_PEAK_ANY_STAGE",
    "OVERALL_SYSTEM_UTILISATION",
    "COHORT_RTT_VALID",
]

pd.DataFrame(
    {
        "no warm-up": {k: baseline[k] for k in KPI_KEYS},
        "1-yr warm-up": {k: with_warmup[k] for k in KPI_KEYS},
    }
).round(2)


,no warm-up,1-yr warm-up
ARRIVED_TOTAL,1307,1302
FLOW_DIAGNOSIS_CONFIRMED,590,593
ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS,0.421237,0.441504
QUEUE_PEAK_ANY_STAGE,4.0,7.0
OVERALL_SYSTEM_UTILISATION,40.650396,37.482485
COHORT_RTT_VALID,True,True


### Stage-level bottleneck view


In [56]:
STAGES = [
    ("screening", "ACCESS_SCREENING_WAIT_DAYS"),
    ("pre_assessment", "ACCESS_PRE_ASSESSMENT_WAIT_DAYS"),
    ("assessment", "ACCESS_ASSESSMENT_WAIT_DAYS"),
    ("further_assessment", "ACCESS_FURTHER_ASSESSMENT_WAIT_DAYS"),
]
pd.DataFrame(
    [
        {
            "stage": s,
            "mean_wait_days": baseline.get(w, 0),
            "peak_queue": baseline.get(f"QUEUE_PEAK_{s.upper()}", 0),
        }
        for s, w in STAGES
    ]
).round(2)


,stage,mean_wait_days,peak_queue
0,screening,0.02,4.0
1,pre_assessment,0.02,2.0
2,assessment,0.05,3.0
3,further_assessment,0.08,2.0


## 6.3 Multiple Replications (Fixed and Random Seeds)

**Fixed seed** (`use_fixed_seed=True`): reproducible experiments, debugging, verification.

**Random seeds** (`use_fixed_seed=False`): statistical variability, confidence intervals.


In [57]:
N_REPS = 5
exp_fixed = Experiment(auditor=Audit(), random_number_set=42)
df_fixed = multiple_runs(
    exp_fixed, n_reps=N_REPS, warmup_days=0, run_length=DEMO_RUN_LENGTH,
    n_jobs=1, use_fixed_seed=True,
)

exp_random = Experiment(auditor=Audit(), random_number_set=42)
df_random = multiple_runs(
    exp_random, n_reps=N_REPS, warmup_days=0, run_length=DEMO_RUN_LENGTH,
    n_jobs=1, use_fixed_seed=False,
)

summary_cols = [
    "ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS",
    "QUEUE_PEAK_ANY_STAGE",
    "OVERALL_SYSTEM_UTILISATION",
]
pd.DataFrame(
    {
        "fixed_seed_mean": df_fixed[summary_cols].mean(),
        "fixed_seed_std": df_fixed[summary_cols].std(),
        "random_seed_mean": df_random[summary_cols].mean(),
        "random_seed_std": df_random[summary_cols].std(),
    }
).round(2)


,fixed_seed_mean,fixed_seed_std,random_seed_mean,random_seed_std
ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS,0.43,0.04,0.43,0.03
QUEUE_PEAK_ANY_STAGE,6.20,1.79,6.00,1.87
OVERALL_SYSTEM_UTILISATION,40.17,3.17,39.46,2.26


## 7.1 Baseline Simulation

Run the default NHS autism pathway with package defaults from `parameters.py`.

The table below reports **per-stage** KPIs for the collection cohort:

| Column | Meaning |
|--------|--------|
| **Referral-to-stage RTT (days)** | Mean days from referral until completing that milestone |
| **Utilisation (%)** | Share of released clinician hours used (collection window) |
| **Peak queue** | Maximum waiting-list depth during collection |
| **Backlog at end** | Patients still waiting in queue when simulation ends |


In [58]:
from adhd_simpy.Model.audit import Audit

baseline_exp = Experiment(auditor=Audit(), random_number_set=42)
baseline_results = single_run(
    baseline_exp, rep=0, warmup_days=WARMUP_DAYS, run_length=RUN_LENGTH
)

# Headline summary
pd.Series(
    {
        "WARMUP_DAYS": baseline_results["WARMUP_DAYS"],
        "RUN_LENGTH": baseline_results["COLLECTION_WINDOW_DAYS"],
        "ARRIVED_TOTAL": baseline_results["ARRIVED_TOTAL"],
        "Diagnosis RTT (days)": baseline_results["ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS"],
        "Peak queue (any stage)": baseline_results["QUEUE_PEAK_ANY_STAGE"],
        "Overall utilisation (%)": baseline_results["OVERALL_SYSTEM_UTILISATION"],
    },
    name="production baseline",
).round(2)


WARMUP_DAYS                 730.00
RUN_LENGTH                 1825.00
ARRIVED_TOTAL              6630.00
Diagnosis RTT (days)          0.42
Peak queue (any stage)        7.00
Overall utilisation (%)      41.34
Name: production baseline, dtype: float64

In [59]:

STAGE_RTT_KEYS = {
    "screening": "ACCESS_REFERRAL_TO_SCREENING_RTT_DAYS",
    "pre_assessment": "ACCESS_REFERRAL_TO_PRE_ASSESSMENT_RTT_DAYS",
    "assessment": "ACCESS_REFERRAL_TO_ASSESSMENT_RTT_DAYS",
    "further_assessment": "ACCESS_REFERRAL_TO_FURTHER_ASSESSMENT_RTT_DAYS",
    "post_diag_clinical": "ACCESS_REFERRAL_TO_POST_DIAG_CLINICAL_RTT_DAYS",
    "post_diag_other": "ACCESS_REFERRAL_TO_POST_DIAG_OTHER_RTT_DAYS",
    "review": "ACCESS_REFERRAL_TO_REVIEW_RTT_DAYS",
}


stage_rows = []
for stage in Audit.STAGE_NAMES:
    key = stage.upper()
    rtt_key = STAGE_RTT_KEYS.get(stage)
    stage_rows.append(
        {
            "stage": stage,
            "referral_to_stage_rtt_days": baseline_results.get(rtt_key) if rtt_key else None,
            "utilisation_pct": baseline_results.get(f"CAPACITY_UTILISATION_{key}", 0) * 100,
            "peak_queue": baseline_results.get(f"QUEUE_PEAK_{key}", 0),
            "backlog_at_end": baseline_results.get(f"QUEUE_BACKLOG_{key}", 0),
            "mean_queue_during_collection": baseline_results.get(f"QUEUE_MEAN_{key}", 0),
        }
    )

# Diagnosis RTT is a pathway milestone (not a capacity stage)
stage_rows.append(
    {
        "stage": "diagnosis (milestone)",
        "referral_to_stage_rtt_days": baseline_results["ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS"],
        "utilisation_pct": None,
        "peak_queue": None,
        "backlog_at_end": None,
        "mean_queue_during_collection": None,
    }
)

stage_baseline = pd.DataFrame(stage_rows).round(2)
stage_baseline


,stage,referral_to_stage_rtt_days,utilisation_pct,peak_queue,backlog_at_end,mean_queue_during_collection
0,screening,0.07,60.19,7.0,0.0,0.06
1,pre_assessment,0.13,37.35,2.0,0.0,0.00
2,assessment,0.35,55.91,4.0,0.0,0.03
3,further_assessment,0.54,30.04,3.0,0.0,0.02
4,post_diag_clinical,NaN,29.73,2.0,0.0,0.02
5,post_diag_other,NaN,20.51,2.0,0.0,0.01
6,review,NaN,22.58,2.0,0.0,0.01
7,diagnosis (milestone),0.42,NaN,NaN,NaN,NaN


## 7.2 Parameter Experiments

Investigate referral demand, capacity, and branching via `Experiment(**kwargs)`.


In [60]:
low = single_run(
    Experiment(auditor=Audit(), random_number_set=42, iat=1 / REFERRALS_PER_DAY),
    rep=0, warmup_days=0, run_length=DEMO_RUN_LENGTH,
)
high = single_run(
    Experiment(auditor=Audit(), random_number_set=42, iat=1 / (REFERRALS_PER_DAY * 2)),
    rep=0, warmup_days=0, run_length=DEMO_RUN_LENGTH,
)
extra_cap = single_run(
    Experiment(auditor=Audit(), random_number_set=42, workforce_hours_assessment=36.0),
    rep=0, warmup_days=0, run_length=DEMO_RUN_LENGTH,
)

compare_keys = [
    "ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS",
    "QUEUE_PEAK_ANY_STAGE",
    "OVERALL_SYSTEM_UTILISATION",
]
pd.DataFrame(
    {
        f"{REFERRALS_PER_DAY:.0f} rpd": {k: low[k] for k in compare_keys},
        f"{REFERRALS_PER_DAY*2:.0f} rpd": {k: high[k] for k in compare_keys},
        "+assessment hours": {k: extra_cap[k] for k in compare_keys},
    }
).round(2)


,5 rpd,10 rpd,+assessment hours
ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS,0.42,44.66,0.40
QUEUE_PEAK_ANY_STAGE,4.00,412.00,4.00
OVERALL_SYSTEM_UTILISATION,40.65,65.58,34.48


## 7.3 Warm-up and Steady-State Analysis

Compare empty system, warm-up, and operational steady state.


In [61]:
warmup_rows = []
for label, warmup in [("empty start", 0), ("1-year warm-up", 365), ("2-year warm-up", 730)]:
    res = single_run(
        Experiment(auditor=Audit(), random_number_set=42),
        rep=0, warmup_days=warmup, run_length=DEMO_RUN_LENGTH,
    )
    warmup_rows.append(
        {
            "scenario": label,
            "warmup_days": warmup,
            "diagnosis_rtt_days": res["ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS"],
            "peak_queue": res["QUEUE_PEAK_ANY_STAGE"],
            "utilisation_pct": res["OVERALL_SYSTEM_UTILISATION"],
        }
    )
pd.DataFrame(warmup_rows).round(2)


,scenario,warmup_days,diagnosis_rtt_days,peak_queue,utilisation_pct
0,empty start,0,0.42,4.0,40.65
1,1-year warm-up,365,0.44,7.0,37.48
2,2-year warm-up,730,0.45,7.0,39.89


## 7.4 Dynamic Calibration

**Status: planned.** Will automatically calibrate provider models by matching operational targets (RTT, queue size, workforce utilisation).

**Current approach:** manual calibration via `devon_parameters.py` — scale workforce hours until simulated RTT matches published ICB statistics (see §7.6).


## 7.5 Freeze-State Initialisation

**Status: planned.** Will capture a simulation snapshot at operational steady state and restore it for scenario analysis.

**Current workaround:** use `warmup_days=730` (or longer) to build a realistic backlog before the collection window.


## 7.6 Scenario Analysis

Evaluate interventions: demand growth, capacity changes, provider-specific calibration.

### Devon (DAANA) — NHS published data

| Parameter | Value | Source |
|-----------|-------|--------|
| Referrals / weekday | **~2.7** | [DAANA](https://www.dpt.nhs.uk/service-details/service/devon-adult-autism-and-adhd-service-daana-20/) — 177/quarter (Dec 2025) |
| Waiting list | **3,305** | DAANA (Dec 2025) |
| Mean wait (ICB) | **~791 days** | [NHS Devon ICB](https://www.dartmouth-today.co.uk/news/patients-with-suspected-autism-referral-in-devon-wait-more-than-two-years-on-average-for-assessment-824203) Jun 2025 |
| NICE target | **91 days** | NICE 13-week RTT |


In [62]:
from adhd_simpy.Model.devon_parameters import (
    DAANA_AUTISM_WAITING_LIST,
    DEVON_ICB_MEAN_WAIT_DAYS,
    DEVON_REFERRALS_PER_WEEKDAY,
    NICE_RTT_TARGET_DAYS,
    devon_experiment,
    devon_run_kwargs,
)

devon_exp = devon_experiment(random_number_set=42)
devon_results = single_run(devon_exp, rep=0, **devon_run_kwargs())

sim_rtt = devon_results["ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS"]
pd.Series(
    {
        "Referrals/weekday (input)": DEVON_REFERRALS_PER_WEEKDAY,
        "DAANA waiting list (published)": DAANA_AUTISM_WAITING_LIST,
        "Simulated diagnosis RTT (days)": sim_rtt,
        "Published Devon ICB mean (days)": DEVON_ICB_MEAN_WAIT_DAYS,
        "NICE target (days)": NICE_RTT_TARGET_DAYS,
        "Gap vs NICE (days)": sim_rtt - NICE_RTT_TARGET_DAYS,
        "Peak queue": devon_results["QUEUE_PEAK_ANY_STAGE"],
        "Patients in system at end": devon_results["IN_SYSTEM_END"],
    },
    name="Devon scenario",
).apply(lambda v: round(v, 1) if isinstance(v, (int, float)) else v)


Referrals/weekday (input)             2.7
DAANA waiting list (published)     3305.0
Simulated diagnosis RTT (days)      709.6
Published Devon ICB mean (days)     791.0
NICE target (days)                   91.0
Gap vs NICE (days)                  618.6
Peak queue                          761.0
Patients in system at end           191.0
Name: Devon scenario, dtype: float64

In [63]:
demand_levels = [5, 10, 15]
scenario_rows = []
for rpd in demand_levels:
    rep_df = multiple_runs(
        Experiment(auditor=Audit(), random_number_set=42, iat=1.0 / rpd),
        n_reps=3, warmup_days=0, run_length=DEMO_RUN_LENGTH, n_jobs=1,
    )
    scenario_rows.append(
        {
            "referrals_per_day": rpd,
            "mean_diagnosis_rtt": rep_df["ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS"].mean(),
            "mean_peak_queue": rep_df["QUEUE_PEAK_ANY_STAGE"].mean(),
            "mean_utilisation_pct": rep_df["OVERALL_SYSTEM_UTILISATION"].mean(),
        }
    )
pd.DataFrame(scenario_rows).round(2)


,referrals_per_day,mean_diagnosis_rtt,mean_peak_queue,mean_utilisation_pct
0,5,0.42,5.33,39.90
1,10,46.76,427.33,62.91
2,15,158.41,1450.00,64.20


---

# 8. Model Verification and Validation

## 8.1 Internal Verification



In [65]:
import importlib
import adhd_simpy.Model.verification as verification
importlib.reload(verification)

from adhd_simpy.Model.verification import (
    run_demand_stress_verification,
    run_flow_conservation_verification,
    run_math_convergence_verification,
    run_rtt_cohort_verification,
    run_seed_verification,
)

run_seed_verification()
run_flow_conservation_verification()
run_rtt_cohort_verification()
run_demand_stress_verification()
run_math_convergence_verification()


 SUITE: 1. SEED CONTROL & REPRODUCIBILITY VERIFICATION
 -> SUCCESS: Fixed seeds are deterministic and reproducible.

 SUITE: 2. PATIENT FLOW MASS-CONSERVATION VERIFICATION
  Arrived: 6509 | Exited: 6509 | In system: 0
 -> SUCCESS: Mass balance verified (arrivals = exits + in system).

 SUITE: 3. RTT COHORT COMPLETENESS VERIFICATION
  Diagnosis RTT samples: 2961 | Diagnoses confirmed: 2961 | In system: 0
 -> SUCCESS: Cohort RTT computed after full drain (zero backlog).

 SUITE: 5. DEMAND STRESS TEST
   5 rpd: peak queue=4 | diagnosis RTT=0.4 d | utilisation=40.7%
  10 rpd: peak queue=412 | diagnosis RTT=44.7 d | utilisation=65.6%
  20 rpd: peak queue=2,518 | diagnosis RTT=272.4 d | utilisation=64.5%
 -> SUCCESS: Queue and RTT increase with demand; utilisation shows expected stress response.

 SUITE: 4. WORKFORCE-HOURS MATHEMATICAL CONVERGENCE
  Assessment RTT: 0.291 d | Diagnosis RTT: 0.371 d | Delta: 0.055
 -> SUCCESS: Infinite-capacity scheduler validation passed.




#
## 8.2 Dynamic Calibration Verification

**Status: planned** — will verify convergence, target matching, and snapshot consistency.

## 8.3 External Validation (Future)

Compare model outputs against NHS provider operational data (RTT, queues, utilisation, throughput).
Devon scenario (§7.6) is a first step toward external validation using published statistics.


---

# 9. Future Work

## 9.1 Provider Calibration Using NHS Operational Data

Use operational NHS provider data to calibrate arrival rates, capacity, branching probabilities, and service durations. Extend `devon_parameters.py` pattern to other ICBs.

## 9.2 Intervention Analysis

Evaluate operational improvements before implementation: additional clinicians, increased clinics, demand management, service redesign.

## 9.3 Model Optimisation

Potential future work: Simulation-Based Optimisation (SBO), multi-parameter calibration, Bayesian optimisation, parallel calibration.

## 9.4 Decision Support System

Potential deployment as an interactive dashboard, provider planning tool, or NHS service planning platform.

